# Selecting Data in a DataFrame

Our goal is to understand how to select a subset of data: some columns, some rows, or both.

**Learning outcomes**

- Peek at a large DataFrame with `head()`, `tail()` and `sample()`.
- Select columns by name (one or several) and rows by slicing.
- Tell `.loc` (labels, end **inclusive**) apart from `.iloc` (positions, end **exclusive**).
- Explain why chained indexing such as `df['col'][7:13]` is discouraged and use `.loc[rows, col]` instead.
- Understand and fix a `DtypeWarning` when reading a file, and use `isin`, `nunique` and `set_index`.

**Before you start:** you should know what a DataFrame, a Series and an index are, and how `read_csv` works (notebook `08_ReadingCSV`).

These notebooks are tested with **pandas 3.x**.

In [ ]:
import pandas as pd
from course_checks import check, check_equal

print(pd.__version__)

**You should see:** a version starting with `3.`.

In [ ]:
import matplotlib.pyplot as plt
from course_viz import BOTH, COL, ROW, compare_slicing, explore_slicing, flow, legend, show_mask, show_selection
from course_viz.intro import cheat_table

We're going to use a new dataset here, to demonstrate how to deal with larger datasets. This is a subset of the 311 service requests from [NYC Open Data](https://nycopendata.socrata.com/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9). "311" is the New York hotline where residents report problems: noise, broken street lights, no heating...

## A warning, and how to fix it

Let's read the file in the simplest way, and look closely at what happens.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    complaints_raw = pd.read_csv('311-service-requests.csv')

for w in caught:
    print(w.category.__name__, ':', w.message)

**You should see:**

```text
DtypeWarning : Columns (8: Incident Zip) have mixed types. Specify dtype option on import or set low_memory=False.
```

(In a plain `read_csv` call, this appears as a red warning box. We captured it here so we can read it calmly.)

**What does it mean?** pandas reads big files in chunks and guesses each column's type chunk by chunk. In column 8, `Incident Zip` (the zip code), most values look like numbers (`11432`) but a few are text, such as `NO CLUE` or `77092-2016`. So some chunks were guessed as numbers and others as text, and the column ends up *mixed*: some values are numbers, some strings. That can silently break later comparisons: `complaints['Incident Zip'] == '11432'` and `== 11432` would each match only part of the data.

**The fix:** tell pandas the type yourself. A zip code is an *identifier*, not a quantity (you never add two zip codes, and leading zeros such as `01234` matter), so text (`str`) is the right type.

In [ ]:
complaints = pd.read_csv('311-service-requests.csv', dtype={'Incident Zip': str})
complaints.shape

**You should see:** `(111069, 52)` and **no warning**: 111,069 complaints, 52 columns. With the type declared, there is nothing left to guess.

Note that in pandas 3 text columns are shown with the dtype `str` (in pandas 2 and earlier you would see `object`). We will meet this in `dtypes`:

In [ ]:
complaints.dtypes.head(10)

**You should see:** `Unique Key` as `int64` and `Created Date`, `Closed Date`, `Agency`, ... all as `str`. Yes, even the dates are plain text for now: turning them into real dates comes in the next notebook.

## Peek

Let's have a peek first at how the DataFrame looks.

In [ ]:
complaints

**You should see:** the first 5 and last 5 rows with `...` in the middle, and the line `111069 rows x 52 columns` at the bottom. Jupyter hides the middle so that the page stays readable.

If you think that it's too long, you can use `head()` for the first rows.

In [ ]:
complaints.head()

Or `tail()` for the last rows.

In [ ]:
complaints.tail()

To see a few *random* rows, which is often more representative than the first rows (data is frequently sorted), use `sample()`. `random_state` makes the choice repeatable.

In [ ]:
complaints.sample(3, random_state=0)[['Agency', 'Complaint Type', 'Borough']]

**You should see:** 3 rows with scattered index numbers (not 0, 1, 2), showing the columns `Agency`, `Complaint Type` and `Borough`. Re-running gives the same rows because of `random_state=0`.

## Selecting columns

We can start by looking at the existing column names.

In [ ]:
complaints.columns

A single column in square brackets gives a `Series`.

In [ ]:
complaints['Complaint Type']

**You should see:** 111069 values with the index `0, 1, 2, ...` on the left, then `Name: Complaint Type, Length: 111069, dtype: str` at the bottom.

In [ ]:
top = complaints['Complaint Type'].value_counts().head(8).sort_values()
ax = top.plot.barh(color='#0072b2', figsize=(8, 3.5), title='Top 8 complaint types (311 calls)')
ax.set_xlabel('Number of complaints')
plt.show()

**What to notice:** a column (a `Series`) summarised by `value_counts()` becomes a ranking. The longest bar is the most common complaint; the counts are on the x-axis.

A **list** of column names gives a DataFrame with only those columns (note the double brackets: the outer ones select, the inner ones build a list).

In [ ]:
complaints[['Complaint Type', 'Borough']].head()

How many different values does a column hold? `nunique()` counts them, and `unique()` lists them.

In [ ]:
print(complaints['Borough'].nunique(), 'boroughs:', complaints['Borough'].unique())
print(complaints['Complaint Type'].nunique(), 'complaint types')

**You should see:** `6 boroughs:` followed by the borough names (including `Unspecified`), then `165 complaint types`.

## Selecting rows: `[]`, `.loc` and `.iloc`

If you care about row-wise selection, a slice inside `[]` takes rows by **position**.

In [ ]:
complaints[:5]

`[:5]` means "from the start up to, but not including, position 5": the first five rows. What if you want rows 7 to 13?

In [ ]:
complaints[7:13]

**You should see:** 6 rows, with index labels 7, 8, 9, 10, 11, 12. Position 13 is *not* included.

Pandas has two explicit tools which are clearer than bare `[]`:

| | `.loc[rows, cols]` | `.iloc[rows, cols]` |
|---|---|---|
| Selects by | **labels** (index values and column names) | **integer positions** (0, 1, 2, ...) |
| Slice end | **inclusive** | **exclusive** (like Python lists) |

Here the index labels happen to equal the positions (0, 1, 2, ...), which makes the difference easy to see: run both.

In [ ]:
by_label = complaints.loc[7:13, ['Complaint Type', 'Descriptor']]
by_position = complaints.iloc[7:13, [5, 6]]   # columns 5 and 6 are 'Complaint Type' and 'Descriptor'

print('loc :', len(by_label), 'rows, labels', by_label.index.min(), 'to', by_label.index.max())
print('iloc:', len(by_position), 'rows, labels', by_position.index.min(), 'to', by_position.index.max())
by_label

**You should see:**

```text
loc : 7 rows, labels 7 to 13
iloc: 6 rows, labels 7 to 12
```

and a table of 7 rows (labels 7 to 13). `.loc` includes the end label 13; `.iloc` stops *before* position 13. Remember: **loc = label, end inclusive; iloc = position, end exclusive.**

Which one to use? Use `.loc` when you know *names* (`'Complaint Type'`, a date), use `.iloc` when you want "the first 3 columns" or "the last row" regardless of names.

### See the difference

Same request, two rules. The green-outlined cells are what each call returns.

In [ ]:
legend(rows_picked=ROW, column_picked=COL, row_and_column=BOTH)
compare_slicing(complaints[['Agency', 'Complaint Type', 'Descriptor']], 7, 13, col='Complaint Type')

**What to notice:** the left table has one more picked row than the right: `.loc` ends *on* label 13, `.iloc` stops *before* position 13.

Now drag the sliders and watch the two rules pick rows. Try `start=0, length=3`.

In [ ]:
explore_slicing(complaints[['Agency', 'Complaint Type', 'Descriptor']], 'Complaint Type');

**What to notice:** the two tables always differ by exactly one row, at the bottom.

### Cheat sheet

In [ ]:
cheat_table(
    ['', 'selects by', 'slice end', 'example'],
    [['<code>df[...]</code>', 'column name(s), or row <b>positions</b> in a slice', 'exclusive', "<code>df['a']</code>, <code>df[7:13]</code>"],
     ['<code>df.loc[...]</code>', '<b>labels</b> (index values, column names)', '<b>inclusive</b>', "<code>df.loc[7:13, 'a']</code>"],
     ['<code>df.iloc[...]</code>', '<b>integer positions</b>', 'exclusive', '<code>df.iloc[7:13, 0:2]</code>']])
flow('df', '.loc[ rows , columns ]', 'rows AND columns picked at once', caption='One pair of brackets, rows first, then columns')

**What to notice:** only `.loc` includes its end. Rows always come before columns, separated by a comma.

### Rows and one column: avoid chained indexing

What if you want rows 7 to 13 of `Complaint Type` only? The clean way is a single `.loc` with rows *and* column:

In [ ]:
complaints.loc[7:13, 'Complaint Type']

You will often see this alternative:

In [ ]:
complaints['Complaint Type'][7:13]

**You should see:** the same kind of Series, but with only 6 values (7 to 12!). The first has 7 values (7 to 13). Same idea, different result, because the second one is `[]` with a position slice (end exclusive) applied after the column was selected.

This second style is called **chained indexing**: two selections in a row. It is discouraged because:

1. it is easy to get confused about which rule (label or position) applies, as you just saw;
2. when you later *assign* (`df['col'][7:13] = ...`) a chained selection may modify a temporary copy instead of your DataFrame. In pandas 3 this never changes your DataFrame (pandas uses *copy-on-write*) and prints a `ChainedAssignmentError` warning; older versions gave a confusing `SettingWithCopyWarning` and sometimes worked, sometimes not;
3. `.loc[rows, col]` states what you want in one step.

Rule of thumb: **one pair of brackets with `.loc[rows, columns]`.**

The two styles pictured step by step. In the chained version the second step works on an *intermediate object*, not on your DataFrame.

In [ ]:
flow('complaints', "['Complaint Type']  (step 1: a new Series)", '[7:13]  (step 2: positions, end excluded)', caption='Chained: two steps, two rules')
flow('complaints', ".loc[7:13, 'Complaint Type']", 'one step, one rule (labels, end included)', caption='Clean: one step')

A tiny demo of the assignment problem. We try to set the first two values of `a` to 0 through a chained selection:

In [ ]:
import warnings

demo = pd.DataFrame({'a': [1, 2, 3, 4], 'b': [5, 6, 7, 8]})
print('before:', demo['a'].tolist())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    demo['a'][0:2] = 0              # chained: changes only a temporary Series
print('after chained:', demo['a'].tolist(), '<- unchanged!')
print('pandas said  :', caught[0].category.__name__ if caught else 'nothing')

demo.loc[0:1, 'a'] = 0              # one step: really modifies demo
print('after .loc    :', demo['a'].tolist())

**You should see:** `after chained: [1, 2, 3, 4]` (the edit was lost, with a `ChainedAssignmentError` warning captured), then `after .loc    : [0, 0, 3, 4]`.

**What to notice:** the chained assignment wrote into a temporary object that was thrown away (copy-on-write), while `.loc[rows, col] = ...` changed `demo` itself.

## Selecting multiple columns and multiple rows

It is possible to select several rows and several columns at once with `.loc`.

In [ ]:
complaints.loc[7:13, ['Complaint Type', 'Descriptor']]

In [ ]:
show_selection(complaints[['Agency', 'Complaint Type', 'Descriptor', 'Borough']].iloc[:16],
               rows=list(range(7, 14)), cols=['Complaint Type', 'Descriptor'],
               caption='.loc[7:13, [Complaint Type, Descriptor]]  ->  7 rows x 2 columns')
show_selection(complaints[['Agency', 'Complaint Type', 'Descriptor', 'Borough']].iloc[:16],
               rows=list(range(7, 13)), cols=['Complaint Type', 'Descriptor'],
               caption='.iloc[7:13, [5, 6]]  ->  6 rows x 2 columns')

**What to notice:** blue rows cross yellow columns; the green outlined block where they meet is exactly what you get back. The second picture has one blue row fewer.

What if you want rows 7 to 12 *and* rows 100 to 199 in one go? We can build a boolean mask, a `True`/`False` per row, and combine the two parts with `|` (or).

In [ ]:
idx1 = (complaints.index < 13) & (complaints.index >= 7)
idx2 = (complaints.index < 200) & (complaints.index >= 100)

idx = idx1 | idx2

complaints.loc[idx, ['Complaint Type', 'Descriptor']]

**You should see:** a table of 106 rows (`106 rows x 2 columns`): labels 7 to 12, then 100 to 199. Masks are the heart of filtering; the next notebook is all about them.

## `isin`: is the value in this list?

To select rows whose value is one of several, `isin` is shorter than chaining many `==` with `|`. It returns a `True`/`False` per row, which `.loc` understands.

In [ ]:
nypd_or_dot = complaints.loc[complaints['Agency'].isin(['NYPD', 'DOT']), ['Agency', 'Complaint Type']]
print(nypd_or_dot.shape)
nypd_or_dot['Agency'].unique()

**You should see:** `(33002, 2)` and the two agencies `NYPD` and `DOT` (in some order).

In [ ]:
show_mask(complaints, complaints['Agency'].isin(['NYPD', 'DOT']), ['Agency', 'Complaint Type'], n=8, label="Agency isin ['NYPD','DOT']")

**What to notice:** `isin` gives one `True`/`False` per row (green &#10003; vs orange &#10007;); `.loc` keeps only the green ones.

## The index: what does "set index" mean?

The index is the set of row labels. Right now it is just the numbers 0, 1, 2... which carry no meaning. If one column uniquely identifies each row, we can make it the index with `set_index`. Then `.loc` finds rows by that identifier.

In [ ]:
by_key = complaints.set_index('Unique Key')
by_key[['Agency', 'Complaint Type']].head(3)

**You should see:** the same rows as before, but `Unique Key` (numbers like `26589651`) is now printed in bold on the left instead of `0, 1, 2`, and is no longer an ordinary column. `set_index` returned a *new* DataFrame; `complaints` itself did not change. Use `.reset_index()` to move the index back to a normal column.

Now we can look up a row by its complaint number:

In [ ]:
first_key = complaints.loc[0, 'Unique Key']
by_key.loc[first_key, ['Agency', 'Complaint Type', 'Borough']]

**You should see:** a Series of 3 values (the agency, complaint type and borough of the very first row), named by the key.

### ✏️ Your turn

Use `.loc` to select the `'Agency'` and `'Borough'` columns for the rows with labels `100` to `104` (inclusive!). Store the result in `five_rows`.

In [ ]:
five_rows = ...  # <- replace ... with complaints.loc[..., [...]]

In [ ]:
check_equal(lambda: five_rows.shape, (5, 2), hint="complaints.loc[100:104, ['Agency', 'Borough']] (loc includes the end label)")
check_equal(lambda: five_rows.index.tolist(), [100, 101, 102, 103, 104], hint='labels 100 to 104')

<details>
<summary>Show solution</summary>

```python
five_rows = complaints.loc[100:104, ['Agency', 'Borough']]
```

</details>

### ✏️ Your turn

Select the `'Complaint Type'` of the rows whose `Agency` is in the list `['HPD', 'DEP']` (use `isin` inside `.loc`). Store the Series in `hpd_dep` and its length in `n_hpd_dep`.

In [ ]:
hpd_dep = ...    # <- replace ...
n_hpd_dep = ...  # <- replace ... (hint: len(...))

In [ ]:
check_equal(lambda: n_hpd_dep, 47964, hint="complaints.loc[complaints['Agency'].isin(['HPD', 'DEP']), 'Complaint Type']")

<details>
<summary>Show solution</summary>

```python
hpd_dep = complaints.loc[complaints['Agency'].isin(['HPD', 'DEP']), 'Complaint Type']
n_hpd_dep = len(hpd_dep)
```

</details>

### ✏️ Your turn

Using `.iloc`, select the first 4 rows and the first 3 columns of `complaints` into `corner`. Then find out how many different `Agency` values exist with `nunique()` and store it in `n_agencies`.

In [ ]:
corner = ...      # <- replace ... with complaints.iloc[..., ...]
n_agencies = ...  # <- replace ...

In [ ]:
check_equal(lambda: corner.shape, (4, 3), hint='complaints.iloc[0:4, 0:3] (iloc excludes the end)')
check_equal(lambda: n_agencies, 26, hint="complaints['Agency'].nunique()")

<details>
<summary>Show solution</summary>

```python
corner = complaints.iloc[0:4, 0:3]
n_agencies = complaints['Agency'].nunique()
```

</details>

## ✅ Check yourself

- I can explain why `read_csv` printed a `DtypeWarning` and how `dtype={'Incident Zip': str}` fixes it.
- I know `.loc` selects by label with the end *included*, and `.iloc` by position with the end *excluded*.
- I can select one column (Series) or several (`df[['a', 'b']]`, a DataFrame).
- I can say why `df['col'][7:13]` is discouraged and write `df.loc[7:13, 'col']` instead.
- I can use `isin`, `nunique`, `sample` and `set_index`.

## Next

In `10_Filter` you will keep only the rows that match conditions, using boolean masks, `isin`, `str.contains` and handling of missing values.